# Task 3 — Kafka Topic Design

## Yêu cầu của đề

Có bốn topic tách biệt cho node, edge, metadata và parser error. Mỗi event mang
schema version và event time.

## Thiết kế và lý do

Node/edge/metadata dùng cleanup `compact` với record key lần lượt là
`node_id`, `edge_id`, `file_id`; error dùng `delete` và retention bảy ngày.
Mỗi topic một partition để bảo toàn thứ tự trong môi trường lab một broker.
Contract và JSON Schema ở
[`task3/TOPIC_CONTRACT.md`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task3/TOPIC_CONTRACT.md) và
[`task3/schemas/`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task3/schemas).

## Output thật đã chạy

Hai artifact dưới được thu trực tiếp từ Kafka CLI sau khi stack healthy.


In [1]:
from pathlib import Path

print("TOPICS")
print(Path("../artifacts/task3/topics_list.txt").read_text(encoding="utf-8").strip())
print("\nCONFIG SUMMARY")
for line in Path("../artifacts/task3/topics_describe.txt").read_text(encoding="utf-8").splitlines():
    if "PartitionCount:" in line:
        print(line.strip())


TOPICS
cpg.edges
cpg.errors
cpg.metadata
cpg.nodes

CONFIG SUMMARY
Topic: cpg.nodes	TopicId: dce7RYlHRVG5lJ4JkPqw4g	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=compact
Topic: cpg.edges	TopicId: HJDET6eTTvyROBqqG6d08w	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=compact
Topic: cpg.metadata	TopicId: Wn79q10fTq6RDRq0pk5whQ	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=compact
Topic: cpg.errors	TopicId: P92U0eFpScKOgibwRUwiJw	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,cleanup.policy=delete,retention.ms=604800000


## Lệnh quan trọng và bằng chứng

```bash
bash task3/create_topics.sh
bash task3/list_topics.sh
bash task3/describe_topics.sh
bash task3/send_samples.sh
bash task3/consume_samples.sh
```

Compose tạo bốn topic idempotently trong service `kafka-init`; các script vẫn
được giữ để kiểm tra thủ công. Sample JSON đã được validate theo Draft 2020-12.

## Lỗi đã gặp và cách khắc phục

Shell script trên Windows từng có nguy cơ CRLF và Compose từng nội suy nhầm biến
shell. `.gitattributes` ép LF cho `*.sh`; biến chạy trong container được escape
đúng để `docker compose config` không làm mất giá trị.

## Reflection

Topic ownership rõ ràng giúp kiến trúc không vô tình cho node/edge đi qua Spark.
Log compaction là lớp hỗ trợ, còn idempotency cuối cùng vẫn do sink upsert.

## Tái lập

Chạy stack từ root, sau đó chạy năm script trên. JSON sample có thể kiểm bằng
`python -m json.tool` và test contract trong `tests/`.
